# Complete lecture notebook — 0

Read the explanation, predict the result, run the next cell, and interpret its output before continuing. Use **Restart Kernel and Run All** to check that the work does not depend on an old session. The complete chapter narrative, mathematical derivations and physical assumptions are retained around the cells.

## Supplied notebook setup
Select **Julia 1.12**. Start this notebook from its chapter directory. This supplied setup locates the recorded course environment and creates a separate folder for exported figures; it installs no packages. Setup commands are infrastructure, not new assessed language material.

In [1]:
import Pkg
course_root = abspath("../..")
if !isfile(joinpath(course_root, "Project.toml"))
    error("Start this notebook from its week folder, e.g. week-1/notebooks")
end
Pkg.activate(course_root; io=devnull)
output_dir = get(ENV, "COURSE_NOTEBOOK_OUTPUT", joinpath(pwd(), "generated", "lecture-en"))
mkpath(output_dir)
println("Course environment active; figure output directory ready.")

Course environment active; figure output directory ready.


# Intro: Physics in the Age of Computation

## Why Use Computers to Do Physics?

Why do we need programming and computers in physics? They help us follow an idea from a model to a calculation, and from a calculation to measurements we can inspect. We can start with familiar questions and then see how the same approach reaches problems at the frontiers of research.

### Start with measurements and motion

Suppose we measure voltage and current for a conductor. A plot lets us see how the two quantities change together; a straight-line fit gives us an estimate of resistance when the conductor behaves ohmically. In Week 1, we will follow a supplied measurement-to-plot-to-fit example. Later, we will build the fitting method ourselves and examine how the residuals and measurement uncertainty affect the conclusion.

In mechanics, a simple projectile model leads to a parabolic trajectory that we can calculate with familiar equations. What changes when air resistance depends on velocity, or when air density varies with height? Adding these effects couples position and velocity in the equations of motion. Numerical integration advances them in small time steps, letting us compare the ideal path with a more detailed prediction.

### A star disrupted by a black hole

What happens when a star passes too close to a massive black hole? The black hole pulls more strongly on the near side of the star than on the far side. When that tidal difference overcomes the star's self-gravity, the star is disrupted. Its gas stretches into streams that return, collide, heat up, and begin to form an accretion flow.

Analytical estimates tell us where disruption begins and set characteristic orbital timescales. To follow the returning streams as they collide, form shocks, and radiate, researchers solve coupled hydrodynamic and radiation equations numerically across three-dimensional space and time. The image below shows the resulting projected column density of such a calculation carried out by two members of our department; its color scale traces where the disrupted gas is concentrated. Such simulations also predict how the event's light changes as the flow develops and allows us to make predictions completely impossible with analytic calculations.

![image](../../assets/figures/tde-density-middle-left-panel.png)

*The projected log column density during a simulated tidal disruption event; positions are measured in solar radii. From E. Steinberg and N. C. Stone, "Stream--disk shocks as the origins of peak light in tidal disruption events," Nature 625, 463--467 (2024), Fig. 1, [doi:10.1038/s41586-023-06875-y](https://doi.org/10.1038/s41586-023-06875-y).*

### Following a Higgs candidate at CERN

At CERN, the Higgs boson is created in very rare particle collisions and decays almost immediately. The detectors at CERN record millions of collision events. For each one they have to calculate the dynamics of all the particles involved from the measured energies and directions of the particles produced. This is a completely impossible task for normal humans, which is why computerized analysis is necessary.

The image below shows one particular event which produced two photons, and which is a very likely candidate for the formation and decay of the Higgs particle. This particle was theorized to exist in order to explain how some particles received their mass.

![image](../../assets/figures/cern-cms-higgs-diphoton-event.png)

*A CMS diphoton candidate from the Higgs search, drawn from reconstructed detector signals. Image: CMS/CERN, [CERN Higgs boson overview](https://home.cern/science/physics/higgs-boson).*

These two examples show us how computers can be used to analyze data and model systems in ways that are qualitatively and quantitatively different, offering more tools for us physicsts.

## What Is a Program? What Is a Computer?

A *program* is a sequence of instructions that tells a computer how to carry out a task. For our projectile, it could start with a position and velocity, calculate the acceleration, and update the motion at successive times. To write those instructions, we need to say which data to store, how to calculate with them, and when to repeat or stop. A human reader can often fill in a missing step in a derivation; a computer needs that step spelled out.

A *computer* is a physical machine that repeatedly executes these instructions. Its central processor (CPU) carries out arithmetic and logical operations; memory (RAM) holds the data and instructions currently in use; persistent storage keeps files after the program ends; and input/output devices connect the machine to files, screens, instruments, and networks. When we run a Julia script, the Julia language system translates our instructions into operations the CPU can execute, while the operating system manages files, memory, and access to the screen.

As we read a program, we can follow the data from the starting values through the calculations to the result. An *algorithm* describes the steps of that calculation; the physical model helps us interpret what comes out. We will build up this way of working gradually. Week 2 introduces types, input conversion and decisions; Week 3 introduces loops and stopping rules; Week 5 introduces function definitions.

## Why Start with Julia?

Julia was designed for numerical and scientific work. It has a direct notation for vectors, matrices, linear algebra, multiple dispatch, and differential-equation solvers, while still being suitable for exploratory scripts and notebooks. It is also fast enough that the first readable version of a numerical model is often already useful.

For us as physicists, the important first benefit is that the code can look like the model. A vector of measurements, a matrix that represents a linear transformation, and an array of values sampled in time are ordinary Julia objects. The distinction between a single value, a vector, and an element-by-element calculation is visible in the notation. That makes dimensions, indices, units, and the intended physical operation easier to inspect before a numerical mistake becomes buried in a long program.

Julia is expressive because its notation lets us state an operation close to the mathematics, and composable because well-defined operations and package tools can be combined into a larger calculation. A matrix operation can feed into another calculation, for example, while the same numerical idea can be defined for ordinary numbers, vectors, matrices, or later for values that carry units. Julia selects the appropriate method from the types of the inputs; this is called *multiple dispatch*. We will use scientific tools first and return to this design idea when it helps explain how one operation can work across several physical settings.

Julia projects can record the packages and versions used for an analysis in project files. This makes it practical to return to a calculation months later or share it with a collaborator without silently changing the computational tools underneath it. The language also has a strong standard library for numerical linear algebra and a growing ecosystem for differential equations, optimization, statistics, units, plotting, machine learning and scientific data. We will build the core methods ourselves first, then use those packages as carefully checked implementations of ideas we already understand.

Speed matters in physics because it lets us test a model, inspect a result, change an assumption, and repeat. The figure below summarizes the Julia project's microbenchmark suite. Its horizontal axis compares execution time with C on a logarithmic scale, so points further left are faster; its vertical axis is the length of the benchmark implementation. Julia lies close to compiled languages in these small numerical kernels while its example code remains compact. These timings describe the selected kernels; the speed of a full program also depends on its algorithm, data layout, allocations, and how we measure it. The data and programs are available in the [Julia microbenchmark repository](https://github.com/JuliaLang/Microbenchmarks).

![image](../../assets/figures/microbenchmarks-speed-vs-code-length.png)

Most importantly for this course, Julia allows code to follow the mathematical structure closely. Suppose a linear dynamical model is written as $$\mathbf{x}_{n+1}=A\mathbf{x}_n+B\mathbf{u}_n,
\qquad
\mathbf{y}_n=C\mathbf{x}_n.$$ The Julia implementation is almost identical:

**Before running:** predict the value, printed text, or graph described above. This is the corresponding example from the written chapter.

### Supplied values for the notation preview
The written example uses symbolic matrices and vectors. We supply small, dimensionless values so it can actually run. Matrix manipulation is a later topic; this is an unassessed illustration of the notation.

In [2]:
A = [1.0 0.1; 0.0 1.0]
B = [0.0; 0.1;;]
C = [1.0 0.0]
x = [2.0, 3.0]
u = [4.0];

In [3]:
x_next = A * x + B * u
y = C * x

1-element Vector{Float64}:
 2.0

**Check:** compare the result with the explanation below. A cell displays only its final expression automatically; `println` deliberately writes text. Rerun prerequisite cells when changing an input.

Here `x`, `u`, and `y` are vectors, while `A`, `B`, and `C` are matrices. We can compare each line of code directly with its equation. As we do, checking the matrix dimensions and physical units helps us see whether the calculation expresses the model we intended.

There are practical details to get used to. The first run of new Julia code can take longer while Julia compiles specialized machine code; later runs may be much faster. Its community is also smaller than Python's, so the available packages can influence our choice of tool. We will learn to measure performance, and we will use Python when its ecosystem suits the task.

Julia also supports LaTeX-style Unicode input in the read--evaluate--print loop (REPL) and VS Code. To write a scalar expression such as $$v_1^{\prime}\cdot(A v_2),$$ type a backslash command and press `Tab` after each symbol:

- `\alpha<Tab>` produces $\alpha$.

- `\cdot<Tab>` produces $\cdot$.

- `v\_1<Tab>\prime<Tab> \cdot<Tab> (A * v\_2<Tab>)` produces $\mathrm{v}_{1}^{\prime}\mathbin{\cdot}(\mathrm{A} * \mathrm{v}_{2})$.

This produces the mathematical form shown above: variable names can carry the same Greek letters, subscripts, primes, and operators that appear in the calculation.

#### Reading the equations alongside the code

Try matching a line of code to the corresponding step in the model. This is often a useful way to spot a sign error, a misplaced index, or incompatible dimensions, and to check which assumptions the calculation uses.

## Why Python?

Python is the dominant general-purpose language of the scientific data ecosystem. Its libraries are widely used for numerical arrays, plotting, optimization, astronomy, machine learning, symbolic algebra, image analysis, and experimental control. We will introduce Python after the core computational ideas are established, so that we can recognize the same scientific workflow in a second language.

Python is particularly valuable because it connects an analysis to the wider research environment. A laboratory may provide a Python script for an instrument, a collaborator may share a Jupyter notebook, or a public archive may publish an example using a specialized Python library. Learning Python lets us read and adapt those workflows, understand what each step contributes, and check the result. The language is also useful beyond numerical physics: file handling, web services, automation, databases, and data visualization use the same basic tools.

Much numerical Python code sends the heavy calculations to compiled libraries such as NumPy and SciPy. This allows a short Python expression to perform substantial numerical work efficiently. A loop written in ordinary Python and a calculation handled by a numerical library can therefore have quite different running times. When we examine performance, we will look at where the work happens and measure the time it takes.

Python is also readable enough that a short analysis often reads like an instruction to a colleague. The listing below is a supplied preview, not a lesson: we have not yet taught Python syntax, and this code is not assessed. Read it to see the shape of an analysis we can later revisit. Here `dropna` removes rows with missing entries in the named columns, `** 2` squares a column element by element, and `np.polyfit(..., 1)` fits a first-degree polynomial and returns its coefficients, so `fit[0]` is the slope:

### Python example
This chapter also uses Python. The exact code below is executable in the linked **Python companion**; it is displayed here to keep the lecture narrative complete. A notebook has one kernel, so do not paste Python into this Julia kernel.

[Python companion](lecture-python-en.ipynb)

```python
clean = data.dropna(subset=["time_s", "position_m"])
clean["time_squared"] = clean["time_s"] ** 2
fit = np.polyfit(clean["time_squared"], clean["position_m"], 1)
gravity = 2 * fit[0]
```

The names are doing scientific work: the reader can see which columns were used, which transformation was made, and why the fitted slope is related to gravity.

With so many Python packages available, we will sometimes find several tools for the same task. We will choose well-documented ones, understand how they apply to our model, and record the package environment so that we can return to the analysis later. Adapting a library example becomes much easier when we can explain what each step does and check its result.

### A complete example: the signal at both LIGO sites

GW150914 was the first direct detection of gravitational waves and the first observed binary black-hole merger. We can compare the public Hanford and Livingston strain (i.e., how much space itself was distorted at two locations in the USA) around the event with the following Python example, adapted from the [GWpy GW150914 filtering example](https://gwpy.github.io/docs/3.0.7/examples/signal/gw150914/). It fetches 32 seconds from each site, applies the same 50--250 Hz band-pass and 60, 120, and 180 Hz notch filters, and displays the central 0.6 seconds.

The code below is an \"as is\" preview: we have not yet learned the Python syntax or the GWpy library, and none of it is assessed this semester. Its here to get an idea what the workflow in Python looks like, and to demonstrate how extensive the Python ecosystem is. One can find packages for almost anything.

A list comprehension such as `[filter_design.notch(f, hdata.sample_rate) for f in (60, 120, 180)]` builds one filter entry per frequency; the star in `concatenate_zpks(bp, *notches)` unpacks that list into separate arguments; and `lfilt *= -1` multiplies the series in place by minus one.

The filters, the time shift, and the sign flip are the operations that make the two traces comparable.

### Python example
This chapter also uses Python. The exact code below is executable in the linked **Python companion**; it is displayed here to keep the lecture narrative complete. A notebook has one kernel, so do not paste Python into this Julia kernel.

[Python companion](lecture-python-en.ipynb)

```python
from gwosc.datasets import event_gps
from gwpy.timeseries import TimeSeries
from gwpy.signal import filter_design
from gwpy.plot import Plot
t0 = event_gps("GW150914")
hdata = TimeSeries.fetch_open_data("H1", t0 - 16, t0 + 16)
ldata = TimeSeries.fetch_open_data("L1", t0 - 16, t0 + 16)
bp = filter_design.bandpass(50, 250, hdata.sample_rate)
notches = [filter_design.notch(f, hdata.sample_rate) for f in (60, 120, 180)]
zpk = filter_design.concatenate_zpks(bp, *notches)
hfilt = hdata.filter(zpk, filtfilt=True)
lfilt = ldata.filter(zpk, filtfilt=True)
lfilt.shift("6.9ms")
lfilt *= -1
plot = Plot(figsize=(12, 4))
ax = plot.gca()
ax.plot(hfilt, label="LIGO-Hanford", color="gwpy:ligo-hanford")
ax.plot(lfilt, label="LIGO-Livingston", color="gwpy:ligo-livingston")
ax.set(xlim=(t0 - 0.4, t0 + 0.2), ylim=(-1e-21, 1e-21),
       title="GW150914 at the two LIGO sites", ylabel="Amplitude [strain]")
ax.set_xscale("seconds", epoch=t0)
ax.legend()
plot;
```

![image](../../assets/figures/gw150914-gwpy-output.png)

This is the actual overlay returned by these commands. Each downloaded series has 131,072 samples at 4,096 Hz over 32 seconds; the wider interval keeps filter-edge transients far from the displayed event window. The Livingston series is shifted forward by 6.9 ms, the measured inter-site delay reported for this event, and multiplied by $-1$ to account for the detectors' relative orientations. These operations make the waveform shapes directly comparable; they are stated explicitly rather than fitted here. The trailing semicolon suppresses a duplicate notebook representation; in a standalone script we can call `plot.show()` or save the figure. The reproduction script and recorded versions are in `graphics/generate_gwpy_example.py` and `graphics/gw150914-gwpy-output.json`.

#### GW150914: what does this graph measure?

Two black holes merged and emitted gravitational waves. A passing wave changes the relative arm length in an interferometer: the strain $h=\Delta L/L$ is dimensionless. The detector output contains noise as well as any signal. The band-pass suppresses frequencies outside 50--250 Hz, the notch filters suppress narrow mains harmonics, and forward-backward filtering avoids an artificial phase delay. Filtering does not by itself establish a detection. Agreement between two independently located detectors after the documented delay and orientation correction is much stronger evidence than one trace alone, but a real analysis also quantifies noise significance and compares waveform models.

GW150914 was observed on 14 September 2015 and announced in 2016. Both LIGO Hanford and Livingston observed the event; Virgo was not observing then. The 2017 Nobel Prize in Physics was awarded to Rainer Weiss, Barry C. Barish, and Kip S. Thorne for decisive contributions to the LIGO detector and the observation of gravitational waves. Calibration, noise, filtering, and waveform assumptions constrain the interpretation. The public GWOSC strain is distributed under CC BY 4.0; see <https://gwosc.org/GW150914/>.

## Why Learn Both?

Once we have worked through a physical calculation in Julia, much of it will feel familiar when we encounter it in Python. The model and its units stay the same, even as the notation for storing data or calling a library changes. A case with a known answer gives us a starting check in either language; plots and residuals help us explore the result. Learning a second language gives us a chance to recognize these ideas in another setting.

We begin with Julia because its numerical notation lets us follow the structure of a physical calculation closely. Later, Python will open up the larger ecosystem we encounter in laboratories, research groups, and public scientific software. By the end of the course, we should be comfortable reading code in both languages, choosing a tool for a particular task, and explaining the physics behind the calculation.

## How the Year Is Organized

Semester A is primarily a Julia semester. We begin immediately with a supplied laboratory workflow for inspecting measurements, fitting a straight line, plotting residuals, and interpreting parameters with units; independent file and table work follows in Week 7. We then build the programming foundations and numerical methods needed to perform such analyses independently. In Week 12, the Tycho Brahe--Kepler investigation combines these skills in one reproducible analysis.

Semester B introduces Python by transferring methods we already understand, then extends them to eigenproblems, Monte Carlo simulation, nonlinear and uncertainty-aware fitting, Fourier analysis, partial differential equations, performance, machine learning, and symbolic workflows. Weekly assignments provide required practice and readiness checks; computerized assessments place greater weight on solving an unfamiliar problem independently and explaining the physical meaning of the result.

## What We Will Learn

To turn a physics problem into a program, we first need to decide how the calculation will proceed. Which values do we start with? What changes at each step, and when should we stop? We will learn to express those choices through *algorithmic thinking*, choosing a representation for the data and breaking the calculation into steps we can follow and test. Simple cases and physical expectations will help us make sense of the output.

We will then use those habits to:

- read experimental and observational data from files;

- visualize data, models, residuals, and uncertainty;

- fit linear and nonlinear models and interpret their parameters;

- solve equations, integrate functions, and solve ordinary differential equations numerically;

- work with vectors, matrices, Fourier transforms, random processes, and simple PDEs;

- write readable, testable, reproducible scientific code in Julia and Python.

## Teaser: Repeating Kepler's Mars Analysis

At the end of the first semester we will use observations associated with Tycho Brahe's Mars programme to reconstruct a version of Kepler's argument. This data set will become evidence for testing Kepler's first law: planetary orbits are ellipses with the Sun at one focus. We will take observations made at different dates, turn tabulated angular positions into a common geometric description, fit competing orbital models, and show quantitatively why an ellipse succeeds where a circle fails.

The plot below shows 923 measurements of Mars's apparent declination, its north--south angle in the sky, from 1582--1600. We can already see a pattern in how that angle changes over time. To connect it to the shape of the orbit, we will need to account for the viewing geometry and compare the predictions of circular and elliptical models. The declination graph on its own leaves that question open.

![image](../../assets/figures/tycho-mars-declination.png)

Tycho recorded dates and measured positions in tables. Turning those entries into a graph gives us a first view of the observations; building an orbital model will help us understand what produces the pattern. Along the way, we will draw on ideas from the courses you are taking alongside this one:

- **Mechanics:** central forces, orbital motion, and the physical interpretation of an ellipse with the Sun at a focus;

- **Mathematics:** vectors, coordinate transformations, conic sections, matrices, and least-squares fitting;

- **Laboratory:** uncertainty, residuals, data provenance, and the distinction between a good-looking curve and a supported conclusion.

By the time we reach this investigation, we will have the tools to write the analysis ourselves and use historical observations to examine the central empirical claim of Kepler's first law. For now, the graph gives us a question to carry through the semester. We will develop the geometry and fitting methods when we are ready to use them.

### Obtaining the course repository

The student repository is `nshaviv/physprog`. We open a terminal in a parent directory of our choice and run:

### Terminal instructions
Run these in a terminal when setting up the course, **not** as Julia code. They are included for coverage; Run All does not clone a repository or install packages.

In [ ]:
git clone https://github.com/nshaviv/physprog.git
cd physprog

If Git is not installed, open the repository page and use *Code* $\to$ *Download ZIP*, then unpack the archive and open the folder; the remaining steps are identical.

Git creates the local course folder. We run the environment commands from this new repository root and open it in VS Code. The student README records which lectures, homework and solutions have been released. Detailed installation and update instructions are in `STUDENT_SETUP.md` at the repository root. The notebooks can be read from their saved outputs; opening them interactively in VS Code with the Jupyter extension or in JupyterLab is optional. Our personal working location need not match the instructor's computer.

## The course environment: install once, run reproducibly

Install Julia 1.12 using <https://julialang.org/install/>; install Visual Studio Code and its official Julia extension using <https://www.julia-vscode.org/docs/stable/gettingstarted/>. In a new terminal, confirm that the command found on our path is the intended Julia:

### Terminal instructions
Run these in a terminal when setting up the course, **not** as Julia code. They are included for coverage; Run All does not clone a repository or install packages.

In [ ]:
julia --version

Reference output in the written notes (compare with the actual cell output):

```text
julia version 1.12.6
```

Confirm that the output begins with `julia version 1.12`. If another version appears, restart the terminal after installation and check which Julia installation it finds.

Open the repository root in VS Code. This is the folder that contains `Project.toml` after cloning. Run the commands below from that root. The option `--project=.` activates the recorded course environment; the following `-e` option tells Julia to evaluate its argument.

### Terminal instructions
Run these in a terminal when setting up the course, **not** as Julia code. They are included for coverage; Run All does not clone a repository or install packages.

In [ ]:
julia --project=. -e 'using Pkg; Pkg.instantiate()'

Julia reads `Project.toml` and the recorded manifest, downloads any missing recorded packages, and reports package installation or precompilation. Run this once while connected to the network. Do not put `Pkg.add` or `Pkg.update` in course work: those commands would change an environment instead of using the recorded one.

Then run the course verification command with the same project selection:

### Terminal instructions
Run these in a terminal when setting up the course, **not** as Julia code. They are included for coverage; Run All does not clone a repository or install packages.

In [ ]:
julia --project=. verify_environment.jl

Reference output in the written notes (compare with the actual cell output):

```text
Course Julia environment verified.
```

The script checks the configured environment and ends with its success message. It may create temporary plot artifacts outside the repository. If it reports a missing package, close old Julia REPLs, make sure VS Code has selected the course environment, and repeat the instantiate command.

In VS Code, select this same course environment for the Julia extension. A file run from the editor sends values and `println` text to the integrated Julia REPL. The editor is where we write code; the REPL is the running Julia session that evaluates it. This distinction becomes useful in Chapter 1.

### Optional: running the notebooks in VS Code
Every notebook already contains its saved outputs, so we can read it without running anything. To run one interactively:

1.  In VS Code, install the *Jupyter* extension (the *Julia* extension stays for `.jl` files).

2.  Open the repository root, then open a `.ipynb` file and use *Select Kernel* to choose `Julia 1.12` for the lecture and recitation notebooks, or the course Python environment for the Chapter 0 Python companions.

3.  Run cells one at a time, or use *Restart Kernel and Run All* to reproduce a chapter from a clean start.

The same notebooks also open in JupyterLab. Running them is an option, not a requirement.

## Small examples and checks mentioned in the chapter
The following cells make the chapter’s inline examples and worked checks executable. They add no homework solutions.

Inspect both vectors from the notation preview.

In [4]:
(x_next, y)

([2.3, 3.4], [2.0])

Check the selected Julia version.

In [5]:
VERSION

v"1.12.6"

## Finish and reflect
Explain one result in words, name its units or assumptions where relevant, and identify a check that could reveal a mistake. The recitation, homework, optional bonus and Going further descriptions above explain the next steps. Optional background does not become an examination prerequisite.